In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import shutil
import subprocess
from datetime import datetime

DATASET_VERSION = 'charades_sta_3fps_v1'

RUNTIME_ROOT = Path('/content/momentlens_prepare')
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/momentlens_datasets') / DATASET_VERSION

DATA_DIR = RUNTIME_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
VIDEO_DIR = DATA_DIR / 'videos'
ANNOT_DIR = DATA_DIR / 'annotations'
SPLIT_DIR = RUNTIME_ROOT / 'splits'
META_DIR = DATA_DIR / 'metadata'
REPORT_DIR = DATA_DIR / 'reports'

for path in [RAW_DIR, VIDEO_DIR, ANNOT_DIR, SPLIT_DIR, META_DIR, REPORT_DIR, DRIVE_DATASET_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('Runtime root:', RUNTIME_ROOT)
print('Drive dataset root:', DRIVE_DATASET_ROOT)


Mounted at /content/drive
Runtime root: /content/momentlens_prepare
Drive dataset root: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1


In [2]:
def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True)

archive_name = 'videos_3fps_480_noaudio.tar.gz'
runtime_archive = RAW_DIR / archive_name
drive_archive = DRIVE_DATASET_ROOT / 'raw' / archive_name
drive_archive.parent.mkdir(parents=True, exist_ok=True)

run(['wget', '-O', str(RAW_DIR / 'charades_sta_train.txt'),
     'https://raw.githubusercontent.com/26hzhang/VSLNet/master/data/dataset/charades/charades_sta_train.txt'])
run(['wget', '-O', str(RAW_DIR / 'charades_sta_test.txt'),
     'https://raw.githubusercontent.com/26hzhang/VSLNet/master/data/dataset/charades/charades_sta_test.txt'])
run(['wget', '-O', str(RAW_DIR / 'charades.json'),
     'https://raw.githubusercontent.com/26hzhang/VSLNet/master/data/dataset/charades/charades.json'])

run(['wget', '-O', str(RAW_DIR / 'durations.json'),
     'https://huggingface.co/datasets/yeliudev/VideoMind-Dataset/resolve/main/charades_sta/durations.json'])

if drive_archive.exists() and not runtime_archive.exists():
    print('Copying video archive from Drive to runtime:', drive_archive)
    shutil.copy2(drive_archive, runtime_archive)
elif not runtime_archive.exists():
    run(['wget', '-O', str(runtime_archive),
         'https://huggingface.co/datasets/yeliudev/VideoMind-Dataset/resolve/main/charades_sta/videos_3fps_480_noaudio.tar.gz'])
else:
    print('Runtime video archive already exists:', runtime_archive)

print('Archive path:', runtime_archive)
print('Archive size GB:', round(runtime_archive.stat().st_size / (1024**3), 2))


+ wget -O /content/momentlens_prepare/data/raw/charades_sta_train.txt https://raw.githubusercontent.com/26hzhang/VSLNet/master/data/dataset/charades/charades_sta_train.txt
+ wget -O /content/momentlens_prepare/data/raw/charades_sta_test.txt https://raw.githubusercontent.com/26hzhang/VSLNet/master/data/dataset/charades/charades_sta_test.txt
+ wget -O /content/momentlens_prepare/data/raw/charades.json https://raw.githubusercontent.com/26hzhang/VSLNet/master/data/dataset/charades/charades.json
+ wget -O /content/momentlens_prepare/data/raw/durations.json https://huggingface.co/datasets/yeliudev/VideoMind-Dataset/resolve/main/charades_sta/durations.json
+ wget -O /content/momentlens_prepare/data/raw/videos_3fps_480_noaudio.tar.gz https://huggingface.co/datasets/yeliudev/VideoMind-Dataset/resolve/main/charades_sta/videos_3fps_480_noaudio.tar.gz
Archive path: /content/momentlens_prepare/data/raw/videos_3fps_480_noaudio.tar.gz
Archive size GB: 10.77


In [3]:
extract_marker = VIDEO_DIR / '.extract_done'
if not extract_marker.exists():
    run(['tar', '-xzf', str(runtime_archive), '-C', str(VIDEO_DIR)])
    extract_marker.write_text(datetime.now().isoformat())
else:
    print('Videos already extracted:', VIDEO_DIR)

video_files_abs = {}
video_files_rel = {}
for path in VIDEO_DIR.rglob('*'):
    if path.suffix.lower() in ['.mp4', '.avi', '.mkv', '.webm']:
        video_files_abs[path.stem] = str(path)
        video_files_rel[path.stem] = str(path.relative_to(VIDEO_DIR))

print('video files found:', len(video_files_abs))
print('first items:', list(video_files_rel.items())[:5])


+ tar -xzf /content/momentlens_prepare/data/raw/videos_3fps_480_noaudio.tar.gz -C /content/momentlens_prepare/data/videos
video files found: 9848
first items: [('ENDIE', 'videos_3fps_480_noaudio/ENDIE.mp4'), ('PF7HH', 'videos_3fps_480_noaudio/PF7HH.mp4'), ('TNEEH', 'videos_3fps_480_noaudio/TNEEH.mp4'), ('T9ZNR', 'videos_3fps_480_noaudio/T9ZNR.mp4'), ('CLVGY', 'videos_3fps_480_noaudio/CLVGY.mp4')]


In [4]:
with open(RAW_DIR / 'charades.json', 'r') as f:
    vslnet_metadata = json.load(f)

with open(RAW_DIR / 'durations.json', 'r') as f:
    hf_durations = json.load(f)

metadata_durations = {video_id: float(meta['duration']) for video_id, meta in vslnet_metadata.items()}

def parse_charades_sta(path, split_name):
    samples = []
    with open(path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    for idx, line in enumerate(lines):
        left, query = line.split('##', 1)
        parts = left.split()
        video_id = parts[0]
        start = float(parts[1])
        end = float(parts[2])
        metadata_duration = float(metadata_durations.get(video_id, max(end, 0.0)))

        samples.append({
            'sample_id': f'{split_name}_{idx:06d}',
            'video_id': video_id,
            'query': query.strip(),
            'start': start,
            'end': end,
            'duration': metadata_duration,
            'metadata_duration': metadata_duration,
            'raw_start': start,
            'raw_end': end,
        })

    return samples

train_all_raw = parse_charades_sta(RAW_DIR / 'charades_sta_train.txt', 'train')
test_raw = parse_charades_sta(RAW_DIR / 'charades_sta_test.txt', 'test')

print('official train pairs:', len(train_all_raw))
print('official test pairs:', len(test_raw))
print('example:', train_all_raw[0])


official train pairs: 12408
official test pairs: 3720
example: {'sample_id': 'train_000000', 'video_id': 'AO8RW', 'query': 'a person is putting a book on a shelf.', 'start': 0.0, 'end': 6.9, 'duration': 33.67, 'metadata_duration': 33.67, 'raw_start': 0.0, 'raw_end': 6.9}


In [5]:
import random

SEED = 42
VAL_RATIO = 0.2

video_ids = sorted({sample['video_id'] for sample in train_all_raw})
random.Random(SEED).shuffle(video_ids)

num_val_videos = int(round(len(video_ids) * VAL_RATIO))
val_video_ids = set(video_ids[:num_val_videos])
train_video_ids = set(video_ids[num_val_videos:])

train_raw = [sample for sample in train_all_raw if sample['video_id'] in train_video_ids]
val_raw = [sample for sample in train_all_raw if sample['video_id'] in val_video_ids]

print('official train videos:', len(video_ids))
print('train videos:', len(train_video_ids))
print('val videos:', len(val_video_ids))
print('train pairs:', len(train_raw))
print('val pairs:', len(val_raw))
print('test pairs:', len(test_raw))
print('train + val pairs:', len(train_raw) + len(val_raw))
print('train/val video overlap:', len(train_video_ids & val_video_ids))


official train videos: 5338
train videos: 4270
val videos: 1068
train pairs: 9912
val pairs: 2496
test pairs: 3720
train + val pairs: 12408
train/val video overlap: 0


In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def ffprobe_duration(path):
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())

used_video_ids = sorted(
    {sample['video_id'] for sample in train_raw} |
    {sample['video_id'] for sample in val_raw} |
    {sample['video_id'] for sample in test_raw}
)

missing_video_ids = [video_id for video_id in used_video_ids if video_id not in video_files_abs]
if missing_video_ids:
    raise RuntimeError(f'Missing {len(missing_video_ids)} videos, first: {missing_video_ids[:10]}')

actual_durations = {}
failed_ffprobe = {}

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(ffprobe_duration, video_files_abs[video_id]): video_id for video_id in used_video_ids}
    for future in as_completed(futures):
        video_id = futures[future]
        try:
            actual_durations[video_id] = future.result()
        except Exception as exc:
            failed_ffprobe[video_id] = repr(exc)

print('used videos:', len(used_video_ids))
print('actual durations measured:', len(actual_durations))
print('ffprobe failures:', len(failed_ffprobe))
if failed_ffprobe:
    print(list(failed_ffprobe.items())[:5])


used videos: 6672
actual durations measured: 6672
ffprobe failures: 0


In [7]:
def clean_samples(samples, split_name):
    clean = []
    dropped = []
    clamped = []

    for sample in samples:
        video_id = sample['video_id']
        actual_duration = actual_durations.get(video_id)

        if actual_duration is None:
            dropped.append({**sample, 'drop_reason': 'missing_actual_duration'})
            continue

        start = float(sample['start'])
        end = float(sample['end'])
        raw_end = end

        if end > actual_duration:
            end = actual_duration
            clamped.append({
                'sample_id': sample['sample_id'],
                'video_id': video_id,
                'raw_end': raw_end,
                'clamped_end': end,
                'actual_duration': actual_duration,
            })

        if not start < end:
            dropped.append({**sample, 'drop_reason': 'start_not_less_than_end_after_cleaning', 'actual_duration': actual_duration})
            continue

        clean.append({
            **sample,
            'start': start,
            'end': end,
            'duration': actual_duration,
            'actual_duration': actual_duration,
            'duration_source': 'ffprobe',
        })

    report = {
        'split': split_name,
        'input_samples': len(samples),
        'clean_samples': len(clean),
        'dropped_samples': len(dropped),
        'clamped_samples': len(clamped),
    }
    return clean, dropped, clamped, report

train, train_dropped, train_clamped, train_clean_report = clean_samples(train_raw, 'train')
val, val_dropped, val_clamped, val_clean_report = clean_samples(val_raw, 'val')
test, test_dropped, test_clamped, test_clean_report = clean_samples(test_raw, 'test')

cleaning_report = {
    'train': train_clean_report,
    'val': val_clean_report,
    'test': test_clean_report,
    'total_dropped': len(train_dropped) + len(val_dropped) + len(test_dropped),
    'total_clamped': len(train_clamped) + len(val_clamped) + len(test_clamped),
}

print(json.dumps(cleaning_report, indent=2))


{
  "train": {
    "split": "train",
    "input_samples": 9912,
    "clean_samples": 9912,
    "dropped_samples": 0,
    "clamped_samples": 965
  },
  "val": {
    "split": "val",
    "input_samples": 2496,
    "clean_samples": 2496,
    "dropped_samples": 0,
    "clamped_samples": 220
  },
  "test": {
    "split": "test",
    "input_samples": 3720,
    "clean_samples": 3720,
    "dropped_samples": 0,
    "clamped_samples": 384
  },
  "total_dropped": 0,
  "total_clamped": 1569
}


## 8. Verify Clean Dataset

In [8]:
def verify_samples(samples, split_name):
    bad_time = []
    missing_video = []
    bad_duration = []

    for sample in samples:
        video_id = sample['video_id']
        if not sample['start'] < sample['end']:
            bad_time.append(sample)
        if video_id not in video_files_abs:
            missing_video.append(video_id)
        if sample['end'] > sample['duration'] + 1e-3:
            bad_duration.append(sample)

    report = {
        'samples': len(samples),
        'unique_videos': len({sample['video_id'] for sample in samples}),
        'bad_time': len(bad_time),
        'missing_video': len(set(missing_video)),
        'bad_duration': len(bad_duration),
    }
    print(f'[{split_name}]')
    print(json.dumps(report, indent=2))
    return report

verify_report = {
    'train': verify_samples(train, 'train'),
    'val': verify_samples(val, 'val'),
    'test': verify_samples(test, 'test'),
}

assert verify_report['train']['bad_time'] == 0
assert verify_report['val']['bad_time'] == 0
assert verify_report['test']['bad_time'] == 0
assert verify_report['train']['missing_video'] == 0
assert verify_report['val']['missing_video'] == 0
assert verify_report['test']['missing_video'] == 0
assert verify_report['train']['bad_duration'] == 0
assert verify_report['val']['bad_duration'] == 0
assert verify_report['test']['bad_duration'] == 0
print('Clean dataset verification passed.')


[train]
{
  "samples": 9912,
  "unique_videos": 4270,
  "bad_time": 0,
  "missing_video": 0,
  "bad_duration": 0
}
[val]
{
  "samples": 2496,
  "unique_videos": 1068,
  "bad_time": 0,
  "missing_video": 0,
  "bad_duration": 0
}
[test]
{
  "samples": 3720,
  "unique_videos": 1334,
  "bad_time": 0,
  "missing_video": 0,
  "bad_duration": 0
}
Clean dataset verification passed.


In [9]:
def save_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)

def save_lines(lines, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        f.write('\n'.join(lines))

# Raw parsed split artifacts.
save_json(train_raw, ANNOT_DIR / 'train_raw.json')
save_json(val_raw, ANNOT_DIR / 'val_raw.json')
save_json(test_raw, ANNOT_DIR / 'test_raw.json')

# Clean train-ready artifacts.
save_json(train, ANNOT_DIR / 'train.json')
save_json(val, ANNOT_DIR / 'val.json')
save_json(test, ANNOT_DIR / 'test.json')

save_lines(sorted(train_video_ids), SPLIT_DIR / 'train_video_ids.txt')
save_lines(sorted(val_video_ids), SPLIT_DIR / 'val_video_ids.txt')

save_json(video_files_rel, META_DIR / 'video_manifest_relative.json')
save_json(actual_durations, META_DIR / 'actual_durations.json')
save_json(metadata_durations, META_DIR / 'vslnet_metadata_durations.json')
save_json(hf_durations, META_DIR / 'hf_durations.json')
save_json(cleaning_report, REPORT_DIR / 'cleaning_report.json')
save_json(verify_report, REPORT_DIR / 'verify_report.json')

duration_diffs = [abs(float(metadata_durations[v]) - float(hf_durations[v])) for v in metadata_durations if v in hf_durations]

dataset_manifest = {
    'dataset_version': DATASET_VERSION,
    'created_at': datetime.now().isoformat(),
    'annotation_source': 'https://github.com/26hzhang/VSLNet/tree/master/data/dataset/charades',
    'video_source': 'https://huggingface.co/datasets/yeliudev/VideoMind-Dataset/tree/main/charades_sta',
    'video_archive': archive_name,
    'video_archive_expected_drive_path': str(DRIVE_DATASET_ROOT / 'raw' / archive_name),
    'duration_source_for_training': 'ffprobe actual video duration',
    'model_sampling_fps': 3,
    'max_timesteps': 96,
    'split': {
        'source': 'official_train',
        'val_ratio': VAL_RATIO,
        'split_unit': 'video_id',
        'seed': SEED,
    },
    'counts': {
        'official_train_pairs': len(train_all_raw),
        'train_pairs': len(train),
        'val_pairs': len(val),
        'test_pairs': len(test),
        'official_train_videos': len(video_ids),
        'train_videos': len(train_video_ids),
        'val_videos': len(val_video_ids),
        'test_videos': len({sample['video_id'] for sample in test}),
        'video_files_in_archive': len(video_files_rel),
        'used_videos': len(used_video_ids),
    },
    'cleaning_report': cleaning_report,
    'verify_report': verify_report,
    'duration_metadata_comparison': {
        'common_videos': len(duration_diffs),
        'max_abs_diff_seconds': max(duration_diffs) if duration_diffs else None,
    },
    'artifact_layout': {
        'annotations': 'data/annotations/{train,val,test}.json',
        'splits': 'splits/{train_video_ids,val_video_ids}.txt',
        'metadata': 'data/metadata/',
        'reports': 'data/reports/',
        'video_archive': f'raw/{archive_name}',
    },
}

save_json(dataset_manifest, DATA_DIR / 'dataset_manifest.json')
print(json.dumps(dataset_manifest, indent=2))


{
  "dataset_version": "charades_sta_3fps_v1",
  "created_at": "2026-04-30T02:57:08.132391",
  "annotation_source": "https://github.com/26hzhang/VSLNet/tree/master/data/dataset/charades",
  "video_source": "https://huggingface.co/datasets/yeliudev/VideoMind-Dataset/tree/main/charades_sta",
  "video_archive": "videos_3fps_480_noaudio.tar.gz",
  "video_archive_expected_drive_path": "/content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/raw/videos_3fps_480_noaudio.tar.gz",
  "duration_source_for_training": "ffprobe actual video duration",
  "model_sampling_fps": 3,
  "max_timesteps": 96,
  "split": {
    "source": "official_train",
    "val_ratio": 0.2,
    "split_unit": "video_id",
    "seed": 42
  },
  "counts": {
    "official_train_pairs": 12408,
    "train_pairs": 9912,
    "val_pairs": 2496,
    "test_pairs": 3720,
    "official_train_videos": 5338,
    "train_videos": 4270,
    "val_videos": 1068,
    "test_videos": 1334,
    "video_files_in_archive": 9848,
    "used_vide

In [10]:
def copy_file_to_drive(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print('Already exists, skip:', dst)
        return
    shutil.copy2(src, dst)
    print('Copied:', dst)

def copy_tree_to_drive(src_dir, dst_dir):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for src in src_dir.rglob('*'):
        if src.is_file():
            dst = dst_dir / src.relative_to(src_dir)
            copy_file_to_drive(src, dst)

copy_tree_to_drive(ANNOT_DIR, DRIVE_DATASET_ROOT / 'data' / 'annotations')
copy_tree_to_drive(SPLIT_DIR, DRIVE_DATASET_ROOT / 'splits')
copy_tree_to_drive(META_DIR, DRIVE_DATASET_ROOT / 'data' / 'metadata')
copy_tree_to_drive(REPORT_DIR, DRIVE_DATASET_ROOT / 'data' / 'reports')
copy_file_to_drive(DATA_DIR / 'dataset_manifest.json', DRIVE_DATASET_ROOT / 'data' / 'dataset_manifest.json')
copy_file_to_drive(runtime_archive, DRIVE_DATASET_ROOT / 'raw' / archive_name)

print('Dataset artifact persisted to:', DRIVE_DATASET_ROOT)


Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/annotations/train.json
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/annotations/val_raw.json
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/annotations/val.json
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/annotations/test.json
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/annotations/train_raw.json
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/annotations/test_raw.json
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/splits/train_video_ids.txt
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/splits/val_video_ids.txt
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/metadata/video_manifest_relative.json
Copied: /content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1/data/metada